In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['figure.autolayout'] = True

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, classification_report, roc_auc_score)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
!pip install tensorflow
!pip install intel-tensorflow



RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [9]:
#loading the data

In [10]:
diabetes_df = pd.read_sas('LLCP2024.XPT ', format='xport')

In [11]:
#checking the contents 

In [12]:
diabetes_df.head()

,_STATE,FMONTH,IDATE,IMONTH,IDAY,IYEAR,DISPCODE,SEQNO,_PSU,CTELENM1,...,_LCSCTSN,_LCSPSTF,DRNKANY6,DROCDY4_,_RFBING6,_DRNKWK3,_RFDRHV9,_FLSHOT7,_PNEUMO3,_AIDTST4
0,1.0,2.0,b'02282024',b'02',b'28',b'2024',1100.0,b'2024000001',2.024000e+09,1.0,...,NaN,9.0,2.0,5.397605e-79,1.0,5.397605e-79,1.0,1.0,2.0,2.0
1,1.0,2.0,b'02212024',b'02',b'21',b'2024',1100.0,b'2024000002',2.024000e+09,1.0,...,4.0,9.0,2.0,5.397605e-79,1.0,5.397605e-79,1.0,1.0,1.0,2.0
2,1.0,2.0,b'02212024',b'02',b'21',b'2024',1100.0,b'2024000003',2.024000e+09,1.0,...,4.0,2.0,1.0,1.000000e+02,2.0,1.400000e+03,1.0,NaN,NaN,2.0
3,1.0,2.0,b'02282024',b'02',b'28',b'2024',1100.0,b'2024000004',2.024000e+09,1.0,...,NaN,9.0,2.0,5.397605e-79,1.0,5.397605e-79,1.0,1.0,1.0,2.0
4,1.0,2.0,b'02212024',b'02',b'21',b'2024',1100.0,b'2024000005',2.024000e+09,1.0,...,3.0,9.0,2.0,5.397605e-79,1.0,5.397605e-79,1.0,NaN,NaN,2.0


In [13]:
#displaying the rows and columns

In [14]:
diabetes_df.shape

(457670, 301)

Querying the original dataset. For this project, only certain variables are necessary, so I am subsetting this dataset to only include necessary variables. The necessary variables include: DIABETE4, _RACE, _AGEG5YR, EXERANY2, SMOKE100, GENHLTH, _BMI5CAT, _SEX, and _RFDRHV9. Querying the data helps to block unnecesary noise from the models that will be conducted later on in the analysis, it also helps to summzarize data and find patterns between variables. I am going to first make a list of those necessary videos and then subset them by putting them into a dataframe of their own.

In [15]:
necessary_columns = ['DIABETE4','_RACE','_AGEG5YR','EXERANY2','SMOKE100','GENHLTH','_BMI5CAT','_SEX','_RFDRHV9']
diabetes = diabetes_df[necessary_columns].copy()

Checking the contents of the subsetted dataframe to make sure the transfer was successful.

In [16]:
diabetes.head()

,DIABETE4,_RACE,_AGEG5YR,EXERANY2,SMOKE100,GENHLTH,_BMI5CAT,_SEX,_RFDRHV9
0,3.0,1.0,12.0,1.0,2.0,3.0,2.0,2.0,1.0
1,3.0,1.0,13.0,1.0,1.0,1.0,3.0,1.0,1.0
2,3.0,1.0,8.0,1.0,1.0,2.0,2.0,1.0,1.0
3,3.0,1.0,13.0,1.0,2.0,1.0,3.0,1.0,1.0
4,3.0,1.0,6.0,2.0,2.0,3.0,2.0,1.0,1.0


Displaying the rows and columns of the subsetted data

In [17]:
diabetes.shape

(457670, 9)

This step informs us of the data types we have from our attributes in our dataframe. All of the attributes in our dataframe are 'float64' which means default integers.

In [18]:
diabetes.dtypes

DIABETE4    float64
_RACE       float64
_AGEG5YR    float64
EXERANY2    float64
SMOKE100    float64
GENHLTH     float64
_BMI5CAT    float64
_SEX        float64
_RFDRHV9    float64
dtype: object

This step summarizes the numerical attributes in our dataframe. Even though they are integers, they represent answers of a categories. Informing us of the most chosen (avaerage) category for most of them. The count for each attribute tells inexplicitly the amount of missing values we have, however, we will get an accurate number shortly.

In [19]:
diabetes.describe()

,DIABETE4,_RACE,_AGEG5YR,EXERANY2,SMOKE100,GENHLTH,_BMI5CAT,_SEX,_RFDRHV9
count,457666.000000,457670.000000,457670.000000,457667.000000,428810.000000,457665.000000,414633.000000,457670.000000,457670.000000
mean,2.739718,2.287550,7.781845,1.251427,1.643007,2.659362,3.009244,1.524751,1.869063
std,0.765308,2.511348,3.738928,0.547624,0.667560,1.081163,0.836549,0.499388,2.414067
min,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
25%,3.000000,1.000000,5.000000,1.000000,1.000000,2.000000,2.000000,1.000000,1.000000
50%,3.000000,1.000000,8.000000,1.000000,2.000000,3.000000,3.000000,2.000000,1.000000
75%,3.000000,2.000000,11.000000,1.000000,2.000000,3.000000,4.000000,2.000000,1.000000
max,9.000000,9.000000,14.000000,9.000000,9.000000,9.000000,4.000000,2.000000,9.000000


This step takes the count of the missing values in each attribute. Each of these values are less than 10% of the whole dataset that we have, so decisions later would need to be made on how to handle them.

In [20]:
diabetes.isna().sum()

DIABETE4        4
_RACE           0
_AGEG5YR        0
EXERANY2        3
SMOKE100    28860
GENHLTH         5
_BMI5CAT    43037
_SEX            0
_RFDRHV9        0
dtype: int64

This step is to get counts of each category within each attribute in our susbet dataframe. This lets the researcher know the distribution of the categories and gives them insight into uneven categories.

In [21]:
diabetes['DIABETE4'].value_counts(dropna=False)

DIABETE4
3.0    376125
1.0     65809
4.0     11307
2.0      3395
7.0       798
9.0       232
NaN         4
Name: count, dtype: int64

In [22]:
diabetes['_RACE'].value_counts(dropna=False)

_RACE
1.0    329346
8.0     48646
2.0     35172
4.0     12646
7.0     10486
9.0      9103
3.0      6460
6.0      3737
5.0      2074
Name: count, dtype: int64

In [23]:
diabetes['_AGEG5YR'].value_counts(dropna=False)

_AGEG5YR
10.0    47701
11.0    44774
9.0     43387
13.0    41756
12.0    36803
8.0     34936
7.0     31698
5.0     30899
1.0     29692
6.0     28968
4.0     28804
3.0     26237
2.0     23705
14.0     8310
Name: count, dtype: int64

In [24]:
diabetes['EXERANY2'].value_counts(dropna=False)

EXERANY2
1.0    350061
2.0    106294
7.0       860
9.0       452
NaN         3
Name: count, dtype: int64

In [25]:
diabetes['SMOKE100'].value_counts(dropna=False)

SMOKE100
2.0    258956
1.0    167242
NaN     28860
7.0      2062
9.0       550
Name: count, dtype: int64

In [26]:
diabetes['GENHLTH'].value_counts(dropna=False)

GENHLTH
3.0    156229
2.0    145791
4.0     67919
1.0     64217
5.0     22204
7.0       915
9.0       390
NaN         5
Name: count, dtype: int64

In [27]:
diabetes['_BMI5CAT'].value_counts(dropna=False)

_BMI5CAT
3.0    146563
4.0    139640
2.0    121053
NaN     43037
1.0      7377
Name: count, dtype: int64

In [28]:
diabetes['_SEX'].value_counts(dropna=False)

_SEX
2.0    240163
1.0    217507
Name: count, dtype: int64

In [29]:
diabetes['_RFDRHV9'].value_counts(dropna=False)

_RFDRHV9
1.0    386812
9.0     46698
2.0     24160
Name: count, dtype: int64

This step imputes the missing values using mode imputation for Diabetes, Generalhealth, and Exercise

In [30]:
for col in ['DIABETE4', 'EXERANY2', 'GENHLTH']:
    mode_attr = diabetes[col].mode()[0]
    diabetes[col] = diabetes[col].fillna(mode_attr)

Double checking to make sure our imputation was sucessful.

In [31]:
diabetes.isna().sum()

DIABETE4        0
_RACE           0
_AGEG5YR        0
EXERANY2        0
SMOKE100    28860
GENHLTH         0
_BMI5CAT    43037
_SEX            0
_RFDRHV9        0
dtype: int64

This step converts specific attributes that have categories labeled a '7' or '9' or '14' into missing values so they can be handled properly.

In [32]:
attr = ['SMOKE100', 'DIABETE4', 'EXERANY2','GENHLTH']
diabetes[attr] = diabetes[attr].replace({7: np.nan, 9: np.nan})

cols = ['_RACE', '_RFDRHV9']
diabetes[cols] = diabetes[cols].replace({9: np.nan})

colss = ['_AGEG5YR']
diabetes[colss] = diabetes[colss].replace({14: np.nan})

This step checks to make sure we were successful in creating NaNs in these attributes.

In [33]:
diabetes.isna().sum()

DIABETE4     1030
_RACE        9103
_AGEG5YR     8310
EXERANY2     1312
SMOKE100    31472
GENHLTH      1305
_BMI5CAT    43037
_SEX            0
_RFDRHV9    46698
dtype: int64

This step defines the function we are going to employ for the sochastic categorical imputation.

In [34]:
def impute_by_distribution(df, columns):
    for column in columns:
        dist = df[column].value_counts(normalize=True)
        missing_idx = df[df[column].isna()].index
        if not missing_idx.empty and not dist.empty:
            df.loc[missing_idx, column] = np.random.choice(
                dist.index,
                size=len(missing_idx),
                p=dist.values
            )
    return df

This step imputes majority of the attributes with missing values using sochastic categorical imputation.

In [51]:
attr_to_impute = ['DIABETE4', '_RACE', 'EXERANY2', 'SMOKE100', 'GENHLTH', '_BMI5CAT', '_RFDRHV9']
diabetes = impute_by_distribution(diabetes, attr_to_impute)

This step checks to make sure we have successfully imputated of most of the attributes...

In [52]:
diabetes.isna().sum()

DIABETE4       0
_RACE          0
_AGEG5YR    8310
EXERANY2       0
SMOKE100       0
GENHLTH        0
_BMI5CAT       0
_SEX           0
_RFDRHV9       0
dtype: int64

This step imputes the _AGE5GYR attribute with sochastic categorical imputation using only allowed categories per the data dictionary.

In [53]:
allowed = [7, 8, 9]
dist = diabetes.loc[diabetes['_AGEG5YR'].isin(allowed), '_AGEG5YR'].value_counts(normalize=True)
missing_idx = diabetes[diabetes['_AGEG5YR'].isna()].index
diabetes.loc[missing_idx, '_AGEG5YR'] = np.random.choice(dist.index, size=len(missing_idx), p=dist.values)

This step makes a final check to ensure we have handled all missing values.

In [54]:
diabetes.isna().sum()

DIABETE4    0
_RACE       0
_AGEG5YR    0
EXERANY2    0
SMOKE100    0
GENHLTH     0
_BMI5CAT    0
_SEX        0
_RFDRHV9    0
dtype: int64

This step recodes categories for better perception.

In [55]:
diabetes['DIABETE4'] = diabetes['DIABETE4'].replace({
    1:0,
    2:1,
    3:1,
    4:1
})
diabetes['_AGEG5YR'] = diabetes['_AGEG5YR'].replace({
    1:1,
    2:1,
    3:1,
    4:2,
    5:2,
    6:2,
    7:2,
    8:3,
    9:3,
    10:3,
    11:3,
    12:3,
    13:3
})
diabetes['_SEX'] = diabetes['_SEX'].replace({
    1:0,
    2:1
})
diabetes['EXERANY2'] = diabetes['EXERANY2'].replace({
    1:0,
    2:1
})
diabetes['GENHLTH'] = diabetes['GENHLTH'].replace({
    1:0,
    2:0,
    3:1,
    4:1,
    5:2
})
diabetes['SMOKE100'] = diabetes['SMOKE100'].replace({
    1:1,
    2:0
})
diabetes['_RFDRHV9'] = diabetes['_RFDRHV9'].replace({
    1:0,
    2:1
})
#This specifically reclassifies minoirty counts into another category called 'Other' 
diabetes['_RACE'] = diabetes['_RACE'].replace({
    3:5,
    5:5,
    6:5,
    7:5,
    8:3
})

This step rechecks the dataframe to make sure the recoding was successful.

In [56]:
diabetes.head()

,DIABETE4,_RACE,_AGEG5YR,EXERANY2,SMOKE100,GENHLTH,_BMI5CAT,_SEX,_RFDRHV9
0,1.0,1.0,3.0,0.0,0.0,1.0,2.0,1.0,0.0
1,1.0,1.0,3.0,0.0,1.0,0.0,3.0,0.0,0.0
2,1.0,1.0,3.0,0.0,1.0,0.0,2.0,0.0,0.0
3,1.0,1.0,3.0,0.0,0.0,0.0,3.0,0.0,0.0
4,1.0,1.0,2.0,1.0,0.0,1.0,2.0,0.0,0.0


This step renames the attributes.

In [57]:
diabetes = diabetes.rename(columns={
    'DIABETE4': 'Diabetes',
    '_SEX': 'Sex',
    '_AGEG5YR': 'Age',
    'EXERANY2': 'Exercise',
    '_BMI5CAT': 'BMI',
    'GENHLTH': 'GeneralHealth',
    '_RACE': 'Race',
    'SMOKE100': 'HeavySmoker',
    '_RFDRHV9': 'HeavyAlcohol'
})

This step checks to ensure the attributes have been renamed

In [58]:
diabetes.head()

,Diabetes,Race,Age,Exercise,HeavySmoker,GeneralHealth,BMI,Sex,HeavyAlcohol
0,1.0,1.0,3.0,0.0,0.0,1.0,2.0,1.0,0.0
1,1.0,1.0,3.0,0.0,1.0,0.0,3.0,0.0,0.0
2,1.0,1.0,3.0,0.0,1.0,0.0,2.0,0.0,0.0
3,1.0,1.0,3.0,0.0,0.0,0.0,3.0,0.0,0.0
4,1.0,1.0,2.0,1.0,0.0,1.0,2.0,0.0,0.0


Correlation Heatmap

In [ ]:
cols = ['Diabetes','Race','Age','Exercise','HeavySmoker','GeneralHealth','BMI','Sex','HeavyAlcohol']
corr = diabetes[cols].corr()

fig = plt.figure()
plt.imshow(corr, aspect='auto')
plt.xticks(range(len(cols)), cols, rotation=45)
plt.yticks(range(len(cols)), cols)
plt.title('Correlation Heatmap')
plt.colorbar()
plt.show()

This graph shows the distrubution of Race

In [ ]:
fig = plt.figure()
plt.hist(diabetes['Race'], bins=20) 
plt.title('Histogram of Race')
plt.xlabel('Race')
plt.ylabel('Count')
plt.show()

This graph shows the distribution of BMI

In [ ]:
fig = plt.figure()
plt.hist(diabetes['BMI'], bins=20)  
plt.title('Histogram of BMI')
plt.xlabel('BMI')
plt.ylabel('Count')
plt.show()

This graph shows the comaprison between Age and Diabetes

In [ ]:
sns.countplot(x='Age', hue='Diabetes', data=diabetes)
plt.title('Age vs Diabetes')
plt.show()

This graph shows the comparison between Heavy Alcohol and HeavySmoker

In [ ]:
sns.countplot(x='HeavySmoker', hue='HeavyAlcohol', data=diabetes)
plt.title('HeavySmoker vs HeavyAlcohol')
plt.show()

This graph shows the comparison between Generalhealth and BMI

In [ ]:
sns.countplot(x='GeneralHealth', hue='BMI', data=diabetes)
plt.title('GeneralHealth vs BMI')
plt.show()

Subsetting my data frame for easier deployment

In [59]:
diabetes_sample = diabetes.sample(n=5000, random_state=42)
diabetes_sample.to_csv("diabetes_sample.csv", index=False)

This step splits our data into two sets: a training and testing set. The trainng set is used to train the models on this data and the testing set is employed to see how well the models classify to unseen data.

In [61]:
X = diabetes_sample[['Race', 'GeneralHealth','Sex', 'Exercise', 'HeavySmoker', 'Age',
                       'BMI', 'HeavyAlcohol']]
y = diabetes_sample['Diabetes']


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)
X_train.shape, X_test.shape

((3750, 8), (1250, 8))

This step creates and runs a logistic regression model on our train data and predicts on our testing data to see how well it performs on unseen data. We can see how this model performs on unseen data by using metrics like accuracy, precision, recall, and F-1 score.

In [62]:
logreg_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=500, random_state=RANDOM_STATE))
])

logreg_pipe.fit(X_train, y_train)

y_pred = logreg_pipe.predict(X_test)
y_prob = logreg_pipe.predict_proba(X_test)[:, 1]

print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))


Classification Report:
              precision    recall  f1-score   support

         0.0       0.54      0.07      0.12       187
         1.0       0.86      0.99      0.92      1063

    accuracy                           0.85      1250
   macro avg       0.70      0.53      0.52      1250
weighted avg       0.81      0.85      0.80      1250

ROC-AUC: 0.7739270855866506


In [63]:
rf = RandomForestClassifier(
    n_estimators=50,
    random_state=42,
    class_weight="balanced"   
)

rf.fit(X_train, y_train)


y_pred = rf.predict(X_test)


print("Accuracy:")
print(accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

Accuracy:
0.7128

Classification Report:
              precision    recall  f1-score   support

         0.0       0.29      0.62      0.39       187
         1.0       0.92      0.73      0.81      1063

    accuracy                           0.71      1250
   macro avg       0.60      0.67      0.60      1250
weighted avg       0.82      0.71      0.75      1250

ROC-AUC: 0.7739270855866506


In [124]:
#saving the model in a pickle file

In [66]:
pickle.dump(rf, open("randomforest_model.pkl", "wb"))